# OpenCV + YOLO 实时目标检测系统

## 项目简介

本项目实现一个“上位机视觉检测 + MCU 联动执行”的 YOLO 实时目标检测系统。PC、树莓派或 Jetson Nano 负责摄像头视频采集和 YOLO 模型推理，Arduino 负责接收检测类别、置信度和目标横向位置，并联动蜂鸣器、继电器和指向舵机，形成一套可视化告警与执行原型。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| PC / 树莓派 / Jetson Nano | 1 | 运行 YOLO 模型 |
| 摄像头 | 1 | 视频输入 |
| Arduino Uno | 1 | 执行检测结果联动 |
| 舵机 | 1 | 指向检测目标方向 |
| 蜂鸣器 | 1 | 告警提示 |
| 继电器模块 | 1 | 扩展控制外部设备 |


## 核心功能

- 上位机执行 YOLO 实时目标检测。
- 通过串口发送检测类别、置信度和目标位置。
- Arduino 根据不同类别执行不同联动策略。
- 舵机根据检测框横向位置进行指向。
- 超时自动清除旧检测，避免残留告警状态。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 蜂鸣器 | D3 |
| 状态 LED | D4 |
| 继电器 | D5 |
| 指向舵机 | D9 |
| 上位机串口通信 | USB Serial |


## 接线说明

- 蜂鸣器接 D3，状态 LED 接 D4，继电器控制端接 D5。
- 指向舵机信号线接 D9，舵机电源建议独立供电并与主控共地。
- 上位机通过 USB 串口与 Arduino 通信，保持 `115200` 波特率一致。
- 若继电器用于控制风扇、警灯等外设，需要根据外设电压单独设计供电回路。


## 串口协议

- 检测帧格式：`D,label,confidence,centerX`。
- 示例：`D,person,86,420`。
- 清空帧格式：`CLEAR`，用于上位机显式清除当前告警状态。
- 当前代码默认只接受置信度大于等于 `60` 的检测结果。


## 运行方式

1. 打开 `src/opencv_yolo_real_time_object_detection_system/opencv_yolo_real_time_object_detection_system.ino` 并上传。
2. 上位机部署 YOLO 检测程序，完成视频推理和目标类别识别。
3. 将检测结果按协议通过串口发送给 Arduino。
4. 根据实际画面宽度调整 `FRAME_WIDTH`，确保舵机指向角映射合理。
5. 按演示需求调整不同类别对应的告警逻辑和继电器联动。


## 控制逻辑说明

- `parseDetectionFrame()` 解析检测类别、置信度和横向中心位置。
- 若检测结果置信度不足，则直接忽略，避免噪声检测频繁触发外设。
- `updateOutputs()` 根据当前类别选择联动方式，例如 `fire/smoke` 触发继电器和快速蜂鸣。
- 舵机根据 `centerX` 映射到 `20-160` 度区间，形成“指向目标”的视觉效果。
- 若超过保持时间没有收到新检测，系统自动调用 `clearDetection()` 回到安全静默状态。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 检测结果传输 | 串口能稳定收到类别和位置数据 |
| 告警联动 | `fire`、`smoke`、`person` 触发不同动作 |
| 舵机指向 | 目标左右移动时舵机角度跟随变化 |
| 超时恢复 | 停止发送检测后系统自动清除告警 |


## 可扩展方向

- 增加更多类别与更细粒度的联动策略。
- 将串口链路扩展为 TCP / WiFi 网络通信。
- 叠加 LCD 或 OLED 显示当前检测类别与置信度。
- 与机械臂、门禁、安防等其它 MCU 项目做联动整合。
